# Packet P-045 — Physics-Informed Features

**Decision question:** Do physics-derived features (tolerance factor, octahedral factor, Goldschmidt ratio) improve cross-family generalization? The model currently fails LOGO (tau-b = 0.005, P-037) because composition fractions don't generalize — physics features might.

**Fastest falsifier:** Compute tolerance factor, octahedral factor, and effective ionic radius from existing composition columns. Run LOGO CV with augmented features. If LOGO tau-b stays < 0.02, physics features don't help cross-family.

**Success:** LOGO tau-b ≥ 0.10 with physics features (up from 0.005).
**Kill:** LOGO tau-b < 0.02 — physics features don't break the family wall.

In [1]:
"""Cell 1 — Compute physics-informed features and run LOGO CV."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import LeaveOneGroupOut, KFold
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')

# === Shannon ionic radii (pm, VI coordination) ===
# A-site cations
R_MA = 217   # methylammonium, effective ionic radius in pm
R_FA = 253   # formamidinium
R_Cs = 167   # cesium

# B-site cations
R_Pb = 119   # Pb2+
R_Sn = 110   # Sn2+ (VI coordination)

# X-site anions
R_I = 220    # I-
R_Br = 196   # Br-
R_Cl = 181   # Cl-

def compute_physics_features(row):
    """Compute tolerance factor, octahedral factor, and related physics features
    from composition fractions using Shannon ionic radii."""
    
    # Get composition fractions (already in dataset)
    ma, fa, cs = row.get('MA', 0) or 0, row.get('FA', 0) or 0, row.get('Cs', 0) or 0
    pb, sn = row.get('Pb', 0) or 0, row.get('Sn', 0) or 0
    i_frac, br, cl = row.get('I', 0) or 0, row.get('Br', 0) or 0, row.get('Cl', 0) or 0
    
    # Normalize A-site (handle zero case)
    a_total = ma + fa + cs
    if a_total > 0:
        r_A = (ma * R_MA + fa * R_FA + cs * R_Cs) / a_total
    else:
        r_A = R_MA  # default to MA
    
    # Normalize B-site
    b_total = pb + sn
    if b_total > 0:
        r_B = (pb * R_Pb + sn * R_Sn) / b_total
    else:
        r_B = R_Pb  # default to Pb
    
    # Normalize X-site
    x_total = i_frac + br + cl
    if x_total > 0:
        r_X = (i_frac * R_I + br * R_Br + cl * R_Cl) / x_total
    else:
        r_X = R_I  # default to I
    
    # Goldschmidt tolerance factor: t = (r_A + r_X) / (sqrt(2) * (r_B + r_X))
    # Ideal perovskite: t = 1.0, stable range: 0.8-1.0
    tolerance_factor = (r_A + r_X) / (np.sqrt(2) * (r_B + r_X))
    
    # Octahedral factor: mu = r_B / r_X
    # Stable range: 0.44-0.90
    octahedral_factor = r_B / r_X
    
    # Effective A-site radius (captures cation mixing directly)
    effective_r_A = r_A
    
    # Effective X-site radius (captures halide mixing)
    effective_r_X = r_X
    
    # A-site variance (Goldschmidt variance — higher = more strain)
    if a_total > 0:
        a_site_variance = (ma * (R_MA - r_A)**2 + fa * (R_FA - r_A)**2 + cs * (R_Cs - r_A)**2) / a_total
    else:
        a_site_variance = 0
    
    # X-site variance (halide mixing strain)
    if x_total > 0:
        x_site_variance = (i_frac * (R_I - r_X)**2 + br * (R_Br - r_X)**2 + cl * (R_Cl - r_X)**2) / x_total
    else:
        x_site_variance = 0
    
    # New Goldschmidt ratio (deviation from ideal t=1)
    t_deviation = abs(tolerance_factor - 1.0)
    
    return pd.Series({
        'tolerance_factor': tolerance_factor,
        'octahedral_factor': octahedral_factor,
        'effective_r_A': effective_r_A,
        'effective_r_X': effective_r_X,
        'a_site_variance': np.sqrt(a_site_variance),  # RMS variance
        'x_site_variance': np.sqrt(x_site_variance),
        't_deviation': t_deviation,
    })

# Compute physics features
print("Computing physics features...")
physics_feats = df.apply(compute_physics_features, axis=1)
df = pd.concat([df, physics_feats], axis=1)

# Show distribution of tolerance factor by family
def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

print(f"\n{'=' * 70}")
print("PHYSICS FEATURES BY FAMILY")
print(f"{'=' * 70}")
print(f"{'Family':<20} {'n':>5} {'t_factor':>10} {'oct_factor':>12} {'A_var':>10} {'X_var':>10}")
print("-" * 70)
for fam in ['Pure MA', 'Pure FA', 'MA-FA mixed', 'FA-Cs', 'Triple cation', 'Other']:
    sub = df[df["family"] == fam]
    if len(sub) == 0:
        continue
    print(f"{fam:<20} {len(sub):>5} {sub['tolerance_factor'].mean():>10.4f} "
          f"{sub['octahedral_factor'].mean():>12.4f} "
          f"{sub['a_site_variance'].mean():>10.2f} "
          f"{sub['x_site_variance'].mean():>10.2f}")

# === Feature sets ===
ORIGINAL_FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]

PHYSICS_FEATURES = [
    'tolerance_factor', 'octahedral_factor', 'effective_r_A', 'effective_r_X',
    'a_site_variance', 'x_site_variance', 't_deviation'
]

AUGMENTED_FEATURES = ORIGINAL_FEATURES + PHYSICS_FEATURES

# Replace composition fractions with physics features (test if physics > composition)
PHYSICS_REPLACE_FEATURES = [
    'Perovskite_band_gap',
    'tolerance_factor', 'octahedral_factor', 'effective_r_A', 'effective_r_X',
    'a_site_variance', 'x_site_variance', 't_deviation',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]

TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

y = np.log1p(df[TARGET])

# === LOGO CV comparison ===
logo = LeaveOneGroupOut()
groups = df["family"]

feature_sets = {
    'Original (16 feat)': ORIGINAL_FEATURES,
    'Augmented (16+7)': AUGMENTED_FEATURES,
    'Physics-replace (15)': PHYSICS_REPLACE_FEATURES,
}

print(f"\n{'=' * 70}")
print("LEAVE-ONE-GROUP-OUT CV — ORIGINAL vs PHYSICS FEATURES")
print(f"{'=' * 70}")

logo_results = {}

for name, feats in feature_sets.items():
    X = df[feats].fillna(0)
    
    family_taus = {}
    all_y_true = []
    all_y_pred = []
    
    for train_idx, test_idx in logo.split(X, y, groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        held_out_family = groups.iloc[test_idx].iloc[0]
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, p = kendalltau(y_te, preds)
        family_taus[held_out_family] = tau
        all_y_true.extend(y_te.tolist())
        all_y_pred.extend(preds.tolist())
    
    overall_tau, _ = kendalltau(all_y_true, all_y_pred)
    mean_tau = np.mean(list(family_taus.values()))
    
    logo_results[name] = {
        'overall_tau': overall_tau,
        'mean_family_tau': mean_tau,
        'family_taus': family_taus,
    }
    
    print(f"\n{name}:")
    print(f"  Overall LOGO tau-b:     {overall_tau:.4f}")
    print(f"  Mean per-family tau-b:  {mean_tau:.4f}")
    for fam, tau in sorted(family_taus.items()):
        n_fam = (groups == fam).sum()
        print(f"    {fam:<20} tau-b = {tau:+.4f}  (n={n_fam})")

# === Random CV comparison (sanity check) ===
print(f"\n{'=' * 70}")
print("RANDOM 5-FOLD CV — ORIGINAL vs PHYSICS FEATURES")
print(f"{'=' * 70}")

random_results = {}
N_REPEATS = 4

for name, feats in feature_sets.items():
    X = df[feats].fillna(0)
    all_taus = []
    
    for rep in range(N_REPEATS):
        kf = KFold(n_splits=5, shuffle=True, random_state=rep)
        for tr_idx, te_idx in kf.split(X):
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
            
            params = ET_PARAMS.copy()
            params['random_state'] = rep
            model = ExtraTreesRegressor(**params)
            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)
            
            tau, _ = kendalltau(y_te, preds)
            all_taus.append(tau)
    
    mean_tau = np.mean(all_taus)
    random_results[name] = mean_tau
    print(f"  {name:<25} Random CV tau-b: {mean_tau:.4f}")

# === Feature importance for augmented model ===
print(f"\n{'=' * 70}")
print("FEATURE IMPORTANCE — AUGMENTED MODEL")
print(f"{'=' * 70}")

X_aug = df[AUGMENTED_FEATURES].fillna(0)
model_aug = ExtraTreesRegressor(**ET_PARAMS)
model_aug.fit(X_aug, y)
imp = pd.Series(model_aug.feature_importances_, index=AUGMENTED_FEATURES).sort_values(ascending=False)

for feat, val in imp.items():
    marker = " *** PHYSICS" if feat in PHYSICS_FEATURES else ""
    print(f"  {feat:<45} {val:.4f}{marker}")

Computing physics features...

PHYSICS FEATURES BY FAMILY
Family                   n   t_factor   oct_factor      A_var      X_var
----------------------------------------------------------------------
Pure MA                967     0.9120       0.5412       0.00       0.11
Pure FA                 50     0.9962       0.5283       0.00       0.99
MA-FA mixed            197     0.9705       0.5467      13.48       6.48
FA-Cs                   44     0.9609       0.5462      29.03       4.76
Triple cation          197     0.9657       0.5485      22.34       8.03
Other                   88     0.8167       0.5563       0.64       6.02

LEAVE-ONE-GROUP-OUT CV — ORIGINAL vs PHYSICS FEATURES



Original (16 feat):
  Overall LOGO tau-b:     0.0180
  Mean per-family tau-b:  0.0049
    FA-Cs                tau-b = -0.0488  (n=44)
    MA-FA mixed          tau-b = -0.0724  (n=197)
    Other                tau-b = +0.0566  (n=88)
    Pure FA              tau-b = +0.1420  (n=50)
    Pure MA              tau-b = +0.0783  (n=967)
    Triple cation        tau-b = -0.1264  (n=197)



Augmented (16+7):
  Overall LOGO tau-b:     -0.0115
  Mean per-family tau-b:  0.0044
    FA-Cs                tau-b = +0.0318  (n=44)
    MA-FA mixed          tau-b = -0.1519  (n=197)
    Other                tau-b = +0.1092  (n=88)
    Pure FA              tau-b = +0.0961  (n=50)
    Pure MA              tau-b = +0.0540  (n=967)
    Triple cation        tau-b = -0.1130  (n=197)



Physics-replace (15):
  Overall LOGO tau-b:     0.0044
  Mean per-family tau-b:  -0.0100
    FA-Cs                tau-b = -0.0255  (n=44)
    MA-FA mixed          tau-b = -0.1644  (n=197)
    Other                tau-b = +0.0708  (n=88)
    Pure FA              tau-b = +0.0961  (n=50)
    Pure MA              tau-b = +0.0664  (n=967)
    Triple cation        tau-b = -0.1032  (n=197)

RANDOM 5-FOLD CV — ORIGINAL vs PHYSICS FEATURES


  Original (16 feat)        Random CV tau-b: 0.2960


  Augmented (16+7)          Random CV tau-b: 0.2966


  Physics-replace (15)      Random CV tau-b: 0.2981

FEATURE IMPORTANCE — AUGMENTED MODEL


  Cell_area_measured                            0.1280
  JV_default_Jsc                                0.1132
  JV_default_Voc                                0.1087
  JV_default_FF                                 0.1028
  first_Prvskt_thermal_annealing_time           0.1003
  first_Prvskt_annealing_temperature            0.0809
  Perovskite_thickness                          0.0761
  Perovskite_band_gap                           0.0610
  t_deviation                                   0.0211 *** PHYSICS
  a_site_variance                               0.0207 *** PHYSICS
  tolerance_factor                              0.0207 *** PHYSICS
  octahedral_factor                             0.0192 *** PHYSICS
  MA                                            0.0192
  effective_r_A                                 0.0189 *** PHYSICS
  effective_r_X                                 0.0158 *** PHYSICS
  Br                                            0.0153
  Cs                                            

In [2]:
"""Cell 2 — Evaluate and save."""

# Determine status based on LOGO improvement
orig_logo = logo_results['Original (16 feat)']['overall_tau']
best_logo = max(r['overall_tau'] for r in logo_results.values())
best_name = [k for k, v in logo_results.items() if v['overall_tau'] == best_logo][0]
improvement = best_logo - orig_logo

if best_logo >= 0.10:
    status = "Confirmed"
elif best_logo < 0.02:
    status = "Negative"
else:
    status = "Promising"

# Save results
rows = []
for name, res in logo_results.items():
    for fam, tau in res['family_taus'].items():
        rows.append({
            'feature_set': name,
            'held_out_family': fam,
            'logo_tau_b': tau,
            'overall_logo_tau': res['overall_tau'],
            'mean_family_tau': res['mean_family_tau'],
            'random_cv_tau': random_results.get(name, float('nan')),
        })

pd.DataFrame(rows).to_csv('Packet_P045_Physics_Features.csv', index=False)
print(f"Saved: Packet_P045_Physics_Features.csv")

print(f"\n{'=' * 60}")
print(f"P-045 STATUS: {status}")
print(f"{'=' * 60}")
print(f"Original LOGO tau-b:    {orig_logo:.4f}")
print(f"Best LOGO tau-b:        {best_logo:.4f} ({best_name})")
print(f"Improvement:            {improvement:+.4f}")

if status == "Confirmed":
    print("\nPhysics features break the family wall!")
    print("LOGO tau-b >= 0.10 — cross-family generalization is possible.")
elif status == "Negative":
    print("\nPhysics features do NOT help cross-family generalization.")
    print("The family wall is not about feature representation — it may be")
    print("about fundamentally different degradation mechanisms per family.")
else:
    print(f"\nPhysics features show some improvement but below 0.10 threshold.")
    print("Worth investigating further — may need more physics features")
    print("(decomposition energy, defect formation energy, etc.)")

Saved: Packet_P045_Physics_Features.csv

P-045 STATUS: Negative
Original LOGO tau-b:    0.0180
Best LOGO tau-b:        0.0180 (Original (16 feat))
Improvement:            +0.0000

Physics features do NOT help cross-family generalization.
The family wall is not about feature representation — it may be
about fundamentally different degradation mechanisms per family.
